<a href="https://colab.research.google.com/github/mffcoello/rag-evaluation-pipeline/blob/main/rag_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install langchain langchain-community chromadb sentence-transformers tiktoken

In [13]:
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model to answer questions using external documents.",
    "Vector databases store embeddings — numerical representations of text. ChromaDB is a popular open-source vector database.",
    "Embeddings are dense numerical vectors that capture the semantic meaning of text. Similar texts have embeddings that are close together in vector space.",
    "LangChain is a framework for building applications with language models. It provides tools for document loading, splitting, retrieval, and chaining.",
    "Chunking is the process of splitting large documents into smaller pieces before storing them in a vector database.",
    "Semantic search finds documents based on meaning rather than exact keyword matches. It uses cosine similarity between embedding vectors.",
    "Precision at K (precision@K) measures how many of the top K retrieved documents are actually relevant to the query.",
    "A retrieval pipeline typically has three steps: embed the query, search the vector store, return the top K results.",
    "Fine-tuning a language model updates its weights on new data. RAG is an alternative that keeps the model frozen but gives it access to external knowledge.",
    "Evaluation is critical in RAG systems. Common metrics include precision@K, recall, and answer relevance scores.",
]

print(f"Loaded {len(documents)} documents")


Loaded 10 documents


In [14]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    client=chromadb.EphemeralClient(),
    collection_name="rag_docs"
)

print(f"Stored {len(chunks)} chunks in vector database")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored 10 chunks in vector database


In [15]:
def retrieve(question, k=3):
    results = vectorstore.similarity_search(question, k=k)
    print(f"Question: {question}\n")
    for i, doc in enumerate(results):
        print(f"Result {i+1}: {doc.page_content}\n")
    return results

retrieve("What is RAG and how does it work?")

Question: What is RAG and how does it work?

Result 1: RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model to answer questions using external documents.

Result 2: Fine-tuning a language model updates its weights on new data. RAG is an alternative that keeps the model frozen but gives it access to external knowledge.

Result 3: Evaluation is critical in RAG systems. Common metrics include precision@K, recall, and answer relevance scores.



[Document(metadata={}, page_content='RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model to answer questions using external documents.'),
 Document(metadata={}, page_content='Fine-tuning a language model updates its weights on new data. RAG is an alternative that keeps the model frozen but gives it access to external knowledge.'),
 Document(metadata={}, page_content='Evaluation is critical in RAG systems. Common metrics include precision@K, recall, and answer relevance scores.')]

In [16]:
def evaluate(question, relevant_keywords, k=3):
    results = retrieve(question, k=k)

    # precision@k: how many of the returned results are actually relevant?
    relevant_count = 0
    for doc in results:
        if any(keyword.lower() in doc.page_content.lower() for keyword in relevant_keywords):
            relevant_count += 1

    precision = relevant_count / k
    print(f"Precision@{k}: {precision:.2f} ({relevant_count}/{k} results were relevant)\n")
    return precision

# Test it
evaluate(
    question="What is RAG and how does it work?",
    relevant_keywords=["RAG", "retrieval", "generation", "language model"]
)

Question: What is RAG and how does it work?

Result 1: RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model to answer questions using external documents.

Result 2: Fine-tuning a language model updates its weights on new data. RAG is an alternative that keeps the model frozen but gives it access to external knowledge.

Result 3: Evaluation is critical in RAG systems. Common metrics include precision@K, recall, and answer relevance scores.

Precision@3: 1.00 (3/3 results were relevant)



1.0

In [ ]:
evaluate (question = "how do I store text in a database?", relevant_keywords = ["vector","chromadb","store",])